In [1]:
import random
import networkx as nx
import math
from time import time
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from networkx.algorithms import community

# Define func. to get all network's characteristics and \<s\>

In [ ]:
# SEE COPILOT FOR UNFEASIBLE CENTRALITY AND OTHER MEASURES !!!!

import numpy as np
import networkx as nx
from networkx.algorithms import community


def characterize_network(
    G,
    outfile=None,
    size_density=True,
    degree_stats=True,
    clustering=True,
    shortest_path=True,
    assortativity=True,
    centralities=True,
    communities=True
):

    results = {}

    # ==========================================================
    # 1. Size and density
    # ==========================================================
    if size_density:
        results["N"] = G.number_of_nodes()
        results["L"] = G.number_of_edges()
        results["density"] = nx.density(G)

    # ==========================================================
    # 2. Degree distribution
    # ==========================================================
    if degree_stats:
        degrees = np.array([d for _, d in G.degree()])

        results["k_avg"] = np.mean(degrees)
        results["k_var"] = np.var(degrees)
        results["degrees"] = degrees

    # ==========================================================
    # 3. Clustering coefficient
    # ==========================================================
    if clustering:
        results["clustering"] = nx.average_clustering(G)

    # ==========================================================
    # 4. Average shortest path in GCC
    # ==========================================================
    if shortest_path:
        largest_cc = max(nx.connected_components(G), key=len)
        Gcc = G.subgraph(largest_cc)

        results["gcc_size"] = Gcc.number_of_nodes()
        results["avg_shortest_path"] = nx.average_shortest_path_length(Gcc)

    # ==========================================================
    # 5. Assortativity
    # ==========================================================
    if assortativity:
        results["assortativity"] = nx.degree_assortativity_coefficient(G)

    # ==========================================================
    # 6. Centralities
    # ==========================================================
    if centralities:
        results["degree_centrality"] = nx.degree_centrality(G)

        results["betweenness_centrality"] = nx.betweenness_centrality(G)

        results["eigenvector_centrality"] = nx.eigenvector_centrality(
            G,
            max_iter=1000
        )

    # ==========================================================
    # 7. Communities
    # ==========================================================
    if communities:
        comms = community.louvain_communities(G)

        results["n_communities"] = len(comms)

        results["community_sizes"] = sorted(
            [len(c) for c in comms],
            reverse=True
        )

        results["modularity"] = community.modularity(G, comms)

    # ==========================================================
    # Save to file
    # ==========================================================
    if outfile is not None:

        with open(outfile, "w") as f:

            for key, value in results.items():

                if isinstance(value, dict):
                    f.write(f"{key}\n")
                    for k, v in value.items():
                        f.write(f"    {k}: {v}\n")

                else:
                    f.write(f"{key}: {value}\n")

    return results


############################################
########### function to get <s> ############
############################################

def s_avg(G):
    comps=[]
    G_copy=G.copy()

    for i in range(nx.number_connected_components(G)):
        largest_cluster = max(nx.connected_components(G_copy), key=len)
        comps.append(len(largest_cluster)) 
        G_copy.remove_nodes_from(largest_cluster)

    if len(comps)-1!=0:
        s = (G.number_of_nodes() - comps[0])/(len(comps)-1)
    else:
        s = 0
        
    return s



In [3]:
G = nx.read_edgelist(
    "./data/roadNet-CA.txt",
    create_using=nx.Graph(),
    nodetype=int
)

In [5]:
res = characterize_network(G)

print("Nodes =", res["N"])
print("Edges =", res["L"])
print("Density =", res["density"])

print("<k> =", res["k_avg"])
print("Var(k) =", res["k_var"])

print("Clustering =", res["clustering"])
print("Avg shortest path =", res["avg_shortest_path"])
print("Assortativity =", res["assortativity"])

print("# Communities =", res["n_communities"])
print("Modularity =", res["modularity"])


KeyboardInterrupt: 